# Notebook 5: Hooks

**What you'll learn:**
- What hooks are and why they exist
- All hook event types and when they fire
- How to create a HookProvider (logging, guard, retry)
- How the HookRegistry dispatches events

**Prerequisite:** Complete [NB4_Tools.ipynb](./NB4_Tools.ipynb)

**Companion reading:** `05-hooks.md`

---
## What are Hooks?

Hooks are like **security cameras** placed at key points in the agent's workflow. They watch what happens and can react.

Every time something important happens (model called, tool executed, message added), the SDK fires a **hook event**. Your code can listen for these events and respond.

Common uses:
- **Logging:** Record what the agent does
- **Guarding:** Block dangerous tool calls
- **Retrying:** Retry model calls that fail
- **Metrics:** Track performance and token usage

### The Event Timeline

For a single agent call with one tool use, events fire in this order:

```
agent("hello")
  |
  v
1. BeforeInvocationEvent     -- "Agent is about to start"
  |
  v
2. BeforeModelCallEvent      -- "About to call the AI model"
  |
  v
3. AfterModelCallEvent       -- "Model responded" (can request retry!)
  |
  v
4. MessageAddedEvent         -- "Assistant message added to history"
  |
  v
5. BeforeToolCallEvent       -- "About to call a tool" (can cancel!)
  |
  v
6. AfterToolCallEvent        -- "Tool finished"
  |
  v
7. MessageAddedEvent         -- "Tool result added to history"
  |
  v
--- RECURSE (model called again to see tool result) ---
  |
  v
8. BeforeModelCallEvent      -- "About to call model again"
  |
  v
9. AfterModelCallEvent       -- "Model responded with final answer"
  |
  v
10. MessageAddedEvent        -- "Final message added"
  |
  v
11. AfterInvocationEvent     -- "Agent is done"
```

In [ ]:
# ============================================================
# Look at all available hook event types
# ============================================================

# These are the event classes that hooks can listen for.
# Each represents a specific moment in the agent's lifecycle.

from strands.hooks.events import (
    AgentInitializedEvent,   # Agent was created
    BeforeInvocationEvent,   # Before agent starts processing a call
    AfterInvocationEvent,    # After agent finishes processing a call
    BeforeModelCallEvent,    # Before calling the AI model
    AfterModelCallEvent,     # After model responds (can request retry!)
    BeforeToolCallEvent,     # Before calling a tool (can cancel!)
    AfterToolCallEvent,      # After tool finishes
    MessageAddedEvent,       # When a message is added to history
)

# List them all
events = [
    AgentInitializedEvent,
    BeforeInvocationEvent,
    AfterInvocationEvent,
    BeforeModelCallEvent,
    AfterModelCallEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent,
    MessageAddedEvent,
]

print("=== Hook Event Types ===")
for evt in events:
    print(f"  {evt.__name__}")

---
## Creating a Logging Hook

A **HookProvider** is a class that registers callbacks for specific events. To create one:

1. Create a class with a `register_hooks(self, registry)` method
2. In that method, register callback functions for the events you care about
3. Pass instances to `Agent(hooks=[...])`

**Source:** `src/strands/hooks/registry.py`

In [ ]:
# ============================================================
# Create a LoggingHook that prints every event
# ============================================================

from strands.hooks.registry import HookRegistry, HookProvider
from strands.hooks.events import (
    BeforeInvocationEvent,
    AfterInvocationEvent,
    BeforeModelCallEvent,
    AfterModelCallEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent,
)

class LoggingHook(HookProvider):
    """A hook that prints a message for every lifecycle event."""
    
    def register_hooks(self, registry: HookRegistry, **kwargs):
        """Register callbacks for each event we want to log.
        
        The registry.add_callback() method takes:
          - event_type: which event to listen for
          - callback: the function to call when the event fires
        """
        # Before/after the entire invocation
        registry.add_callback(BeforeInvocationEvent, self.on_before_invocation)
        registry.add_callback(AfterInvocationEvent, self.on_after_invocation)
        
        # Before/after model calls
        registry.add_callback(BeforeModelCallEvent, self.on_before_model)
        registry.add_callback(AfterModelCallEvent, self.on_after_model)
        
        # Before/after tool calls
        registry.add_callback(BeforeToolCallEvent, self.on_before_tool)
        registry.add_callback(AfterToolCallEvent, self.on_after_tool)
    
    def on_before_invocation(self, event: BeforeInvocationEvent, **kwargs):
        print("[HOOK] >>> Agent invocation STARTING")
    
    def on_after_invocation(self, event: AfterInvocationEvent, **kwargs):
        print("[HOOK] <<< Agent invocation COMPLETE")
    
    def on_before_model(self, event: BeforeModelCallEvent, **kwargs):
        print("[HOOK]   > Calling model...")
    
    def on_after_model(self, event: AfterModelCallEvent, **kwargs):
        # AfterModelCallEvent has useful fields:
        #   event.stop_reason -- why the model stopped
        print(f"[HOOK]   < Model responded (stop_reason: {event.stop_reason})")
    
    def on_before_tool(self, event: BeforeToolCallEvent, **kwargs):
        # BeforeToolCallEvent has:
        #   event.tool_name -- which tool is being called
        #   event.tool_input -- the arguments
        print(f"[HOOK]     > Calling tool: {event.tool_name}")
    
    def on_after_tool(self, event: AfterToolCallEvent, **kwargs):
        print(f"[HOOK]     < Tool finished: {event.tool_name}")

print("LoggingHook defined.")

In [ ]:
# ============================================================
# Use the LoggingHook with an agent
# ============================================================

from strands import Agent, tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.
    
    Args:
        city: The name of the city.
    """
    return f"72F and sunny in {city}"

# Create agent with the logging hook and a tool.
# The hook is passed as a list to the hooks parameter.
agent = Agent(
    tools=[get_weather],
    hooks=[LoggingHook()],      # Our custom hook!
    callback_handler=None,       # Quiet mode for cleaner output
)

# Call the agent. Watch the [HOOK] messages in the output.
# You should see the full event timeline:
#   BeforeInvocation -> BeforeModel -> AfterModel -> BeforeTool -> 
#   AfterTool -> BeforeModel -> AfterModel -> AfterInvocation

print("=== Calling agent with LoggingHook ===")
print()
result = agent("What's the weather in Seattle?")
print()
print(f"=== Final result: {result} ===")

### Reading the Output

The hook output above shows the complete event timeline:

1. `>>> Agent invocation STARTING` -- BeforeInvocationEvent
2. `> Calling model...` -- BeforeModelCallEvent (Cycle 1)
3. `< Model responded (stop_reason: tool_use)` -- AfterModelCallEvent (model wants a tool)
4. `> Calling tool: get_weather` -- BeforeToolCallEvent
5. `< Tool finished: get_weather` -- AfterToolCallEvent
6. `> Calling model...` -- BeforeModelCallEvent (Cycle 2, after recursion)
7. `< Model responded (stop_reason: end_turn)` -- AfterModelCallEvent (model is done)
8. `<<< Agent invocation COMPLETE` -- AfterInvocationEvent

---
## Guard Hook -- Blocking Tool Calls

A guard hook can **cancel** a tool call before it executes. This is useful for safety.

The `BeforeToolCallEvent` has a `cancel_tool(message)` method. If you call it, the tool is NOT executed, and the message is returned to the model as the tool result.

In [ ]:
# ============================================================
# Guard Hook: block certain tool calls
# ============================================================

class GuardHook(HookProvider):
    """A hook that blocks tool calls to specific tools."""
    
    def __init__(self, blocked_tools: list):
        # Store the list of tool names to block.
        self.blocked_tools = blocked_tools
    
    def register_hooks(self, registry: HookRegistry, **kwargs):
        # Only listen for BeforeToolCallEvent -- that's where we can cancel.
        registry.add_callback(BeforeToolCallEvent, self.on_before_tool)
    
    def on_before_tool(self, event: BeforeToolCallEvent, **kwargs):
        if event.tool_name in self.blocked_tools:
            # cancel_tool() prevents the tool from executing.
            # The message we pass becomes the tool result that the model sees.
            event.cancel_tool(
                f"Tool '{event.tool_name}' is blocked by security policy."
            )
            print(f"[GUARD] Blocked tool call: {event.tool_name}")

# Define a "dangerous" tool
@tool
def delete_file(filename: str) -> str:
    """Delete a file from the system.
    
    Args:
        filename: The file to delete.
    """
    return f"Deleted {filename}"

# Create agent with the guard hook blocking delete_file
agent_guarded = Agent(
    tools=[get_weather, delete_file],
    hooks=[GuardHook(blocked_tools=["delete_file"])],
    callback_handler=None,
)

# Try to use the blocked tool
print("=== Attempting to use blocked tool ===")
result = agent_guarded("Delete the file named test.txt")
print(f"\nResult: {result}")
# The model should see the block message and respond accordingly.

---
## Retry Hook -- Retrying Model Calls

The `AfterModelCallEvent` has a special power: setting `event.retry = True` tells the SDK to **call the model again**. This is useful for handling transient errors or ensuring quality.

**Warning:** Be careful with retries -- you can create infinite loops. Always limit the number of retries.

In [ ]:
# ============================================================
# Retry Hook: retry model calls based on conditions
# ============================================================

class RetryHook(HookProvider):
    """A hook that retries model calls up to a max count."""
    
    def __init__(self, max_retries: int = 1):
        self.max_retries = max_retries
        self.retry_count = 0
    
    def register_hooks(self, registry: HookRegistry, **kwargs):
        registry.add_callback(AfterModelCallEvent, self.on_after_model)
    
    def on_after_model(self, event: AfterModelCallEvent, **kwargs):
        # Example: retry if the response is very short
        # In real code, you might check for errors or quality issues.
        
        # Get the response text length
        text_length = 0
        if event.message and event.message.get('content'):
            for block in event.message['content']:
                if 'text' in block:
                    text_length += len(block['text'])
        
        if text_length < 10 and self.retry_count < self.max_retries:
            self.retry_count += 1
            event.retry = True  # Tell the SDK to call the model again!
            print(f"[RETRY] Response too short ({text_length} chars). Retrying ({self.retry_count}/{self.max_retries})")
        else:
            self.retry_count = 0  # Reset for next invocation

print("RetryHook defined. This hook retries if the model response is under 10 characters.")

---
## HookRegistry Internals

The `HookRegistry` stores callbacks organized by event type. When an event fires, it dispatches to all registered callbacks for that type.

```python
# Internal structure:
{
    BeforeToolCallEvent: [callback1, callback2],
    AfterModelCallEvent: [callback3],
    ...
}
```

In [ ]:
# ============================================================
# Inspect the HookRegistry
# ============================================================

# Create an agent with both hooks
agent_hooks = Agent(
    tools=[get_weather],
    hooks=[LoggingHook(), GuardHook(blocked_tools=[])],
    callback_handler=None,
)

# The hooks attribute is a HookRegistry
print(f"Type: {type(agent_hooks.hooks).__name__}")
print()

# Look at what's registered
# The registry stores callbacks by event type
print("=== Registered Callbacks ===")
if hasattr(agent_hooks.hooks, '_callbacks'):
    for event_type, callbacks in agent_hooks.hooks._callbacks.items():
        print(f"  {event_type.__name__}: {len(callbacks)} callback(s)")

---
## Hook Events Reference

| Event | When it fires | Key fields | Special powers |
|-------|--------------|------------|----------------|
| `AgentInitializedEvent` | Agent created | `agent` | -- |
| `BeforeInvocationEvent` | Before agent starts | `agent` | -- |
| `BeforeModelCallEvent` | Before model call | `agent`, `messages` | -- |
| `AfterModelCallEvent` | After model responds | `agent`, `message`, `stop_reason` | `retry = True` |
| `BeforeToolCallEvent` | Before tool executes | `agent`, `tool_name`, `tool_input` | `cancel_tool(msg)`, `interrupt()` |
| `AfterToolCallEvent` | After tool finishes | `agent`, `tool_name`, `tool_result` | -- |
| `MessageAddedEvent` | Message added to history | `agent`, `message` | -- |
| `AfterInvocationEvent` | After agent finishes | `agent`, `result` | -- |

**Source:** `src/strands/hooks/events.py`, `src/strands/hooks/registry.py`

---
## Summary

What you learned in this notebook:

- **Hooks** are like security cameras -- they watch and react to lifecycle events
- **8 event types** fire at specific moments (before/after model, before/after tool, etc.)
- **HookProvider** is a class with `register_hooks()` that registers callback functions
- **LoggingHook** prints every event for debugging
- **GuardHook** blocks tool calls using `event.cancel_tool()`
- **RetryHook** retries model calls using `event.retry = True`
- **HookRegistry** stores and dispatches callbacks by event type

**Key source files:**
| File | What it does |
|------|--------------|
| `src/strands/hooks/events.py` | All hook event classes |
| `src/strands/hooks/registry.py` | HookRegistry, HookProvider |

**Next:** [NB6_Messages_And_Conversation.ipynb](./NB6_Messages_And_Conversation.ipynb) -- Message format, conversation management, and context windows